# PMT Average Waveforms & PSD Analysis

Carga desde el cache de preprocessing, aplica filtros de calidad y amplitud,
y produce los plots de análisis de forma de pulso y waveforms promedio.

**Prerequisito:** `pmt_preprocess.py --run <RUN>`

Contenido:
1. Setup: carga + filtrado
2. Qtotal vs Amplitude
3. Qfast vs Qslow (PSD)
4. Qfast/Qslow vs Qslow
5. Clusters de forma de pulso (waveforms promedio por cluster)
6. Qfast/Qslow con colores por cluster
7. Layout PMT — waveforms promedio con deconvolución

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.input_output.pickle_hdf5_reader import WaveformSet_from_hdf5_pickle

from quality_filter import DEFAULT_QUALITY_WINDOWS, DEFAULT_QUALITY_CUTS, build_wfset_quality
from pmt_analysis import (
    compute_qtotal_amplitude,
    compute_qfast_qslow,
    compute_pulse_shape_clusters,
    pmt_average_and_fit,
    plot_pmt_layout,
    TICK_NS,
)

## 1. Setup: carga + filtrado

Reproduce el mismo filtrado que el notebook 02 para que este notebook sea autónomo.
Si ya tienes `wfset` en memoria (por haber ejecutado el notebook 02 en la misma sesión),
puedes saltar esta sección.

In [ ]:
run     = 43363
dettype = 'pmt'

cache_file          = f"cache/wfset_run{run:06d}_{dettype}_ana.hdf5"
wfset_triggered_all = WaveformSet_from_hdf5_pickle(cache_file)
print(f"Cargadas {len(wfset_triggered_all.waveforms)} waveforms")

In [ ]:
# Filtros de calidad (mismos parámetros que en notebook 02)
quality_windows = {**DEFAULT_QUALITY_WINDOWS}
quality_cuts    = {**DEFAULT_QUALITY_CUTS}

wfset_quality, *_ = build_wfset_quality(
    wfset_triggered_all, cuts=quality_cuts, windows=quality_windows
)

In [ ]:
# Filtro adicional de amplitud
def select_amp(waveform: Waveform) -> bool:
    slice_min = slice(78, 83) if waveform.channel == 16 else slice(75, 80)
    baseline  = waveform.analyses['std'].result['baseline']
    y         = waveform.adcs - baseline
    return np.min(y[slice_min]) > 400 and np.max(y[60:80]) < 7e3

wfset = WaveformSet.from_filtered_WaveformSet(wfset_quality, select_amp, show_progress=True)
print(f"wfset final: {len(wfset.waveforms)} waveforms")

## 2. Qtotal vs Amplitude

Cambia `selected_endpoint` y `selected_channel` para explorar otro canal.

In [ ]:
selected_endpoint   = 110
selected_channel    = 16
qtotal_duration_ns  = 1_300.0

qtotal_vals, amplitude_vals = compute_qtotal_amplitude(
    wfset, selected_endpoint, selected_channel,
    qtotal_duration_ns=qtotal_duration_ns,
)

plt.figure(figsize=(8, 6))
plt.scatter(amplitude_vals, qtotal_vals, s=2, alpha=0.15, rasterized=True)
plt.xlabel("Amplitude above baseline [ADC]")
plt.ylabel(f"Qtotal, {qtotal_duration_ns/1e3:.1f} µs integral from peak start [ADC·ticks]")
plt.title(f"Qtotal vs Amplitude | ep {selected_endpoint}, ch {selected_channel}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Qfast vs Qslow (PSD)

In [ ]:
selected_endpoint  = 110
selected_channel   = 14
qfast_duration_ns  = 64.0
qtotal_duration_ns = 1_300.0

qfast_vals, qslow_vals = compute_qfast_qslow(
    wfset, selected_endpoint, selected_channel,
    qfast_duration_ns=qfast_duration_ns,
    qtotal_duration_ns=qtotal_duration_ns,
)

plt.figure(figsize=(8, 6))
plt.scatter(qslow_vals, qfast_vals, s=2, alpha=0.15, rasterized=True)
plt.xlabel(f"Qslow, {qtotal_duration_ns/1e3:.1f} µs integral [ADC·ticks]")
plt.ylabel(f"Qfast, {qfast_duration_ns:.0f} ns integral around peak [ADC·ticks]")
plt.title(f"Qfast vs Qslow | ep {selected_endpoint}, ch {selected_channel}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Qfast/Qslow vs Qslow

In [ ]:
qfast_over_qslow = np.divide(
    qfast_vals, qslow_vals,
    out=np.full_like(qfast_vals, np.nan, dtype=float),
    where=qslow_vals != 0,
)
valid = np.isfinite(qfast_over_qslow)

plt.figure(figsize=(8, 6))
plt.scatter(qslow_vals[valid], qfast_over_qslow[valid], s=4, alpha=0.25, rasterized=True)
plt.xlabel(f"Qslow [ADC·ticks]")
plt.ylabel("Qfast / Qslow")
plt.title(f"Qfast/Qslow vs Qslow | ep {selected_endpoint}, ch {selected_channel}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Clusters de forma de pulso

Define la región del cluster "deviating" en el espacio Qfast-Qslow y compara
las waveforms promedio de cada grupo.

In [ ]:
# Parámetros del cluster — ajusta según lo que ves en los plots de Qfast/Qslow
deviating_qslow_range = (75_000, 175_000)
deviating_qfast_range = (0, 12_500)

records, clusters, corrected_wfs, mean_wfs = compute_pulse_shape_clusters(
    wfset, selected_endpoint, selected_channel,
    deviating_qslow_range=deviating_qslow_range,
    deviating_qfast_range=deviating_qfast_range,
    qfast_duration_ns=qfast_duration_ns,
    qtotal_duration_ns=qtotal_duration_ns,
)
print(f"Deviating: {len(clusters['deviating'])} wf  |  Rest: {len(clusters['rest'])} wf")

time_ns = np.arange(len(mean_wfs["rest"])) * TICK_NS

plt.figure(figsize=(9, 5))
plt.plot(time_ns, mean_wfs["deviating"],
         label=f"Deviating cluster, n={len(clusters['deviating'])}", lw=2)
plt.plot(time_ns, mean_wfs["rest"],
         label=f"Rest, n={len(clusters['rest'])}", lw=2)
plt.axhline(0, color="black", lw=1, alpha=0.5)
plt.xlabel("Time [ns]")
plt.ylabel("Amplitude above baseline [ADC]")
plt.title(f"Average waveforms by cluster | ep {selected_endpoint}, ch {selected_channel}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Qfast/Qslow vs Qslow con colores por cluster

In [ ]:
qslow_arr  = np.array([r["qslow"]           for r in records])
ratio_arr  = np.array([r["qfast_over_qslow"] for r in records])
names_arr  = np.array([r["cluster"]          for r in records])
valid      = np.isfinite(ratio_arr)

plt.figure(figsize=(8, 6))
for cluster_name, color in [("deviating", "tab:orange"), ("rest", "tab:blue")]:
    mask = valid & (names_arr == cluster_name)
    plt.scatter(
        qslow_arr[mask], ratio_arr[mask],
        s=4, alpha=0.25, color=color, rasterized=True,
        label=f"{cluster_name}, n={np.sum(mask)}",
    )
plt.axvspan(*deviating_qslow_range, color="tab:orange", alpha=0.08)
plt.xlabel("Qslow [ADC·ticks]")
plt.ylabel("Qfast / Qslow")
plt.title(f"Qfast/Qslow vs Qslow — clusters | ep {selected_endpoint}, ch {selected_channel}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Layout PMT — waveforms promedio con deconvolución

Genera el grid completo con todos los PMTs. Guarda la figura en `pmts_average_waveforms.png`.

In [ ]:
plot_pmt_layout(wfset, endpoint=110, savefig="pmts_average_waveforms.png")

## Visor interactivo de waveforms por cluster (ipywidgets)

Necesita `ipywidgets`. Ejecutar la celda de abajo para inspeccionar eventos individuales.

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

max_idx = max(len(records) for records in clusters.values()) - 1

def plot_cluster_event(cluster_name, event_index):
    recs = clusters[cluster_name]
    if not recs:
        raise ValueError(f"No waveforms in cluster '{cluster_name}'")
    event_index = min(event_index, len(recs) - 1)
    rec     = recs[event_index]
    wf      = rec["waveform"]
    y       = np.asarray(wf.adcs, dtype=float) - rec["baseline"]
    time_ns = np.arange(len(y)) * TICK_NS

    plt.figure(figsize=(10, 4.5))
    plt.plot(time_ns, y, lw=1.2)
    plt.axhline(0, color="black", lw=1, alpha=0.5)
    plt.axvline(rec["peak_tick"] * TICK_NS, color="tab:red", ls="--", lw=1.2, label="Peak tick")
    plt.axvspan(rec["qfast_start_tick"] * TICK_NS, rec["qfast_stop_tick"]  * TICK_NS,
                color="tab:orange", alpha=0.25, label="Qfast window")
    plt.axvspan(rec["qfast_start_tick"] * TICK_NS, rec["qtotal_stop_tick"] * TICK_NS,
                color="tab:blue",   alpha=0.08, label="Qslow window")
    plt.xlabel("Time [ns]")
    plt.ylabel("ADC - baseline")
    plt.title(
        f"{cluster_name} | event {event_index}/{len(recs)-1} | "
        f"Qfast={rec['qfast']:.0f}, Qslow={rec['qslow']:.0f}, "
        f"Qf/Qs={rec['qfast_over_qslow']:.3f}"
    )
    plt.legend(loc="best")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(
    plot_cluster_event,
    cluster_name=Dropdown(options=["deviating", "rest"], value="deviating", description="Cluster"),
    event_index=IntSlider(min=0, max=max_idx, step=1, value=0, description="Event"),
)